# Milestone 2: how much do wrong labels hold the model back?

Our labels come from Google's cards: two headlines in the same card = same story, in different cards = different stories. The error analysis in `milestone2_experiments.ipynb` showed that Google's cards are sometimes wrong (one story split into two cards) or loose (a digest or opinion piece grouped with a news story). This notebook asks two questions:

1. **Training labels:** can we find the wrong labels in train and does removing them give a better model?
2. **Evaluation labels:** what are the scores when the models are graded against corrected val and test labels instead of Google's?

Steps:
1. **Cross-checking:** train one model on each half of train; each one scores the other half. Pairs where the model strongly disagrees with the label are flagged.
2. **Hand check of the flags:** a sample of flagged pairs is read to see how often the label is really wrong.
3. **Retrain on cleaned train:** drop the kind of flagged pairs that the hand check confirms are mostly wrong; retrain the bi-encoder and the cross-encoder.
4. **Corrected val:** every val pair that at least one model disagrees with is read and given a corrected label. Pairs that every model agrees with are kept as they are.
5. **Corrected test:** the same for test, used once at the end.

The hand checks are saved in `outputs/milestone2/label_noise/*.csv`. They were done by one annotator (an AI assistant) and spot-checked by the team. If a hand-check file is missing or empty, the notebook saves it for reading and skips the steps that need it.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'pairs').is_dir())
sys.path.insert(0, str(ROOT))
from src.m2_common import *          # data, metrics, training and scoring shared by the Milestone 2 notebooks
from src.embeddings import clean_headline

OUT = ROOT / 'outputs' / 'milestone2' / 'label_noise'
OUT.mkdir(parents=True, exist_ok=True)
pairs = load_pairs()
TRAIN, VAL, TEST = pairs['train'], pairs['val'], pairs['test']

def judged(path, column):
    """A hand-check file, or None if it does not exist yet or is not filled in."""
    if not path.exists():
        return None
    d = pd.read_csv(path, keep_default_na=False)
    return d if (d[column].astype(str).str.strip() != '').all() else None

def cached_run(kind, tag, train_fn):
    """Reuse a model trained by an earlier run of this notebook."""
    path = MODELS / 'exp' / tag
    if path.exists():
        print(f'{tag}: using saved model')
        return {'kind': kind, 'path': path, 'prep': clean_headline}
    return train_fn()

print('device:', DEVICE, {s: len(d) for s, d in pairs.items()})

## 1. Cross-checking the training labels

Train is split in two halves by page (snapshot), so the two headlines of a pair always land in the same half. A model trained on one half scores the other half. A pair is **flagged** when the model's score is on the wrong side of its val threshold: a "same story" pair scored below it, or a "different story" pair scored above it. `margin` says how far on the wrong side.

In [ ]:
FLAGS = OUT / 'train_flags.csv'
if FLAGS.exists():
    flags = pd.read_csv(FLAGS, dtype={'timestamp': str})
else:
    snaps = TRAIN.timestamp.unique()
    half = dict(zip(snaps, np.random.default_rng(SEED).permutation(len(snaps)) % 2))
    fold = TRAIN.timestamp.map(half).values
    parts = []
    for k in [0, 1]:
        run = cached_run('bi', f'fold{k}', lambda: train_biencoder(TRAIN[fold == k], f'fold{k}', VAL))
        t = best_threshold(VAL.label.values, score_model(run, VAL))
        other = TRAIN[fold != k].copy()
        other['score'], other['threshold'] = score_model(run, other), t
        parts.append(other)
    flags = pd.concat(parts)
    flags['margin'] = np.where(flags.label == 1, flags.threshold - flags.score, flags.score - flags.threshold)
    flags = flags.sort_values('margin', ascending=False)
    flags.to_csv(FLAGS, index=False)

flags['flagged'] = flags.margin > 0
print(flags.groupby('label').flagged.agg(['sum', 'mean']).rename(index={0: 'different story', 1: 'same story'},
      columns={'sum': 'flagged pairs', 'mean': 'share flagged'}).round(3))

In [ ]:
SAMPLE = OUT / 'train_flags_sample.csv'
if not SAMPLE.exists():
    strong = flags[flags.margin > 0.05]
    sample = pd.concat([strong[strong.label == lab].sample(40, random_state=SEED) for lab in [1, 0]])
    cols = ['label', 'edition', 'score', 'margin', 'headline_a', 'outlet_a', 'headline_b', 'outlet_b', 'timestamp']
    sample[cols].assign(score=lambda d: d.score.round(3), margin=lambda d: d.margin.round(3), label_wrong='', note='').to_csv(SAMPLE, index=False)
    print('saved for hand check:', SAMPLE.relative_to(ROOT))
print(f"pairs with margin > 0.05: same story {((flags.margin > 0.05) & (flags.label == 1)).sum():,}, "
      f"different story {((flags.margin > 0.05) & (flags.label == 0)).sum():,}")

## 2. Hand check of the flagged pairs

40 flagged "same story" pairs and 40 flagged "different story" pairs (margin above 0.05), picked at random. `label_wrong`: `yes` = Google's label is wrong, `no` = the label is right (the model is wrong), `unclear` = same broad topic, different events.

**Rule:** a kind of flagged pair (same-story or different-story) is dropped from train when at least half of its sample has a wrong label (`yes`). Dropping is safer than flipping the label, since `unclear` pairs are common.

In [ ]:
hand = judged(SAMPLE, 'label_wrong')
DROP_SIDES = []
if hand is None:
    print('Hand check not filled in yet:', SAMPLE.relative_to(ROOT))
else:
    hand['label'] = hand.label.map({1: 'same story', 0: 'different story'})
    display(pd.crosstab(hand.label, hand.label_wrong, margins=True))
    share = hand.groupby('label').label_wrong.apply(lambda s: (s == 'yes').mean())
    print((share * 100).round(0).astype(int).astype(str) + '% of flagged pairs have a wrong label')
    DROP_SIDES = [1 if side == 'same story' else 0 for side, s in share.items() if s >= 0.5]
    print('drop flagged pairs with label:', DROP_SIDES or 'none')

## 3. Retraining on the cleaned train set

Same recipes as v1 (bi-encoder) and the cross-encoder in `milestone2_experiments.ipynb`; only the training pairs change.

In [ ]:
RUNS = {'pretrained': {'kind': 'bi', 'path': BASE_MODEL, 'prep': clean_headline},
        'v1': {'kind': 'bi', 'path': MODELS / 'minilm_finetuned', 'prep': clean_headline},
        'cross_encoder': {'kind': 'cross', 'path': MODELS / 'exp' / 'cross_encoder', 'prep': clean_headline}}
if hand is not None:
    drop = flags[(flags.margin > 0.05) & flags.label.isin(DROP_SIDES)]
    key = lambda d: d.timestamp + '|' + d.headline_a + '|' + d.headline_b
    cleaned = TRAIN[~key(TRAIN).isin(set(key(drop)))]
    print(f'cleaned train: {len(cleaned):,} pairs ({len(TRAIN) - len(cleaned):,} dropped)')
    if DROP_SIDES:
        RUNS['v1_cleaned'] = cached_run('bi', 'v1_cleaned', lambda: train_biencoder(cleaned, 'v1_cleaned', VAL))
        RUNS['cross_encoder_cleaned'] = cached_run('cross', 'cross_encoder_cleaned',
                                                   lambda: train_crossencoder(cleaned, 'cross_encoder_cleaned', VAL))
print('models compared:', list(RUNS))

## 4. Corrected val labels

Each model's val threshold is chosen on Google's labels, as before. Every val pair where **at least one** model disagrees with Google's label is saved to `val_check.csv` and read by hand. `same_story`: `yes`, `no`, or `unclear` (dropped from the corrected set). Pairs that all models agree with keep Google's label.

In [ ]:
def disputed(df, scores, thresholds):
    wrong = np.zeros(len(df), int)
    for name, s in scores.items():
        wrong += ((s >= thresholds[name]).astype(int) != df.label.values)
    return wrong

VAL_SCORES = {name: score_model(run, VAL) for name, run in RUNS.items()}
T_GOOGLE = {name: best_threshold(VAL.label.values, s) for name, s in VAL_SCORES.items()}
VAL_CHECK = OUT / 'val_check.csv'
n_wrong = disputed(VAL, VAL_SCORES, T_GOOGLE)
print(f'val pairs disputed by at least one model: {(n_wrong > 0).sum():,} of {len(VAL):,}')
if hand is not None and not VAL_CHECK.exists():   # wait for step 3, so the cleaned models are included
    VAL.assign(pair_id=VAL.index, models_disagreeing=n_wrong)[n_wrong > 0][
        ['pair_id', 'label', 'models_disagreeing', 'edition', 'headline_a', 'outlet_a', 'headline_b', 'outlet_b', 'timestamp']
    ].assign(same_story='', note='').to_csv(VAL_CHECK, index=False)
    print('saved for hand check:', VAL_CHECK.relative_to(ROOT))

In [ ]:
def corrected(df, check):
    """Google's labels with the hand-checked ones replaced; 'unclear' pairs removed."""
    d = df.copy()
    c = check.set_index('pair_id').same_story
    d.loc[c.index[c == 'yes'], 'label'] = 1
    d.loc[c.index[c == 'no'], 'label'] = 0
    return d.drop(index=c.index[c == 'unclear'])

def compare(df, df_clean, scores, thresholds):
    rows = []
    for name, s in scores.items():
        s_clean = s[df.index.get_indexer(df_clean.index)]
        g = evaluate(df.label.values, s, thresholds[name])
        c = evaluate(df_clean.label.values, s_clean, thresholds[name])
        rows.append({'model': name, 'F1 (Google labels)': g['f1'], 'F1 (corrected)': c['f1'],
                     'PR-AUC (Google)': g['pr_auc'], 'PR-AUC (corrected)': c['pr_auc']})
    return pd.DataFrame(rows).set_index('model')

vcheck = judged(VAL_CHECK, 'same_story') if hand is not None else None
if vcheck is None:
    print('val hand check not filled in yet')
else:
    VAL_CLEAN = corrected(VAL, vcheck)
    flips = ((vcheck.same_story == 'yes') & (vcheck.label == 0)) | ((vcheck.same_story == 'no') & (vcheck.label == 1))
    print(f"val: {flips.sum()} labels corrected, {(vcheck.same_story == 'unclear').sum()} unclear pairs removed, "
          f"{len(VAL_CLEAN):,} pairs left -> estimated wrong-label rate {flips.sum() / len(VAL):.1%}")
    # thresholds re-chosen on the corrected val labels, used unchanged on test below
    T_CLEAN = {name: best_threshold(VAL_CLEAN.label.values, s[VAL.index.get_indexer(VAL_CLEAN.index)]) for name, s in VAL_SCORES.items()}
    val_table = compare(VAL, VAL_CLEAN, VAL_SCORES, T_CLEAN)
    val_table.to_csv(OUT / 'val_corrected_scores.csv')
    display(val_table.round(4))

## 5. Corrected test labels (used once)

Same procedure on test, with the thresholds chosen on the corrected val labels.

In [ ]:
TEST_CHECK = OUT / 'test_check.csv'
if vcheck is not None:
    TEST_SCORES = {name: score_model(run, TEST) for name, run in RUNS.items()}
    n_wrong = disputed(TEST, TEST_SCORES, T_CLEAN)
    print(f'test pairs disputed by at least one model: {(n_wrong > 0).sum():,} of {len(TEST):,}')
    if not TEST_CHECK.exists():
        TEST.assign(pair_id=TEST.index, models_disagreeing=n_wrong)[n_wrong > 0][
            ['pair_id', 'label', 'models_disagreeing', 'edition', 'headline_a', 'outlet_a', 'headline_b', 'outlet_b', 'timestamp']
        ].assign(same_story='', note='').to_csv(TEST_CHECK, index=False)
        print('saved for hand check:', TEST_CHECK.relative_to(ROOT))
    tcheck = judged(TEST_CHECK, 'same_story')
    if tcheck is None:
        print('test hand check not filled in yet')
    else:
        TEST_CLEAN = corrected(TEST, tcheck)
        flips = ((tcheck.same_story == 'yes') & (tcheck.label == 0)) | ((tcheck.same_story == 'no') & (tcheck.label == 1))
        print(f"test: {flips.sum()} labels corrected, {(tcheck.same_story == 'unclear').sum()} unclear removed, {len(TEST_CLEAN):,} pairs left")
        test_table = compare(TEST, TEST_CLEAN, TEST_SCORES, T_CLEAN)
        test_table.to_csv(OUT / 'test_corrected_scores.csv')
        display(test_table.round(4))